In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
import re
import numpy as np
import pandas as pd
import re
import gc
import datetime as _dt
import pyarrow.parquet as pq
from collections import defaultdict
from dateutil.relativedelta import relativedelta
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.metrics import precision_recall_curve, auc

In [ ]:
df = pd.read_csv('features_v3.csv')
df.head()

In [ ]:
l = pd.read_csv('cert_dataset/insiders/insiders.csv')
insiders = l[l['dataset']==6.2]['user']
insiders

In [ ]:
df['is_insider'] = df['user'].isin(insiders).astype(int)
df['is_insider'].value_counts()

In [ ]:
X = df.drop(columns='user')

s_scaler = StandardScaler()

m_scaler = MinMaxScaler()

X_scaled_s = s_scaler.fit_transform(X)

X_scaled_m = m_scaler.fit_transform(X)

model = IsolationForest(
    n_estimators=1000,
    max_samples=2000,
    max_features=1.0,
    random_state=42
)

model.fit(X_scaled_s)

In [ ]:
df['anomaly_score'] = -model.score_samples(X_scaled_s)

In [ ]:
sorted_indices = np.argsort(df['anomaly_score'].values)[::-1]

sorted_scores = df['anomaly_score'].values[sorted_indices]

sorted_insiders = df['is_insider'].values[sorted_indices]

insider_plot_indices = np.where(sorted_insiders == 1)[0]

plt.figure(figsize=(10, 6))

plt.plot(range(len(sorted_scores)), sorted_scores, label='Score d\'anomalie', color='#2ecc71', linewidth=2)

plt.scatter(insider_plot_indices, sorted_scores[insider_plot_indices], 
            color='orange', label='Vrais Insiders (Labels)', zorder=5, s=40, edgecolors='black')

plt.title("Méthode du Coude : Détection de la rupture des scores", fontsize=14)
plt.xlabel("Index des observations (triées par score)", fontsize=12)
plt.ylabel("Score d'anomalie", fontsize=12)
plt.legend()
plt.grid(alpha=0.3)

plt.show()